# Metrics Processor Research Notebook

This notebook tests and validates the `metrics_processor.py` module for calculating stock metrics from local price data.

**Workflow:**
1. Load the MetricsProcessor class
2. Get available symbols from data directory
3. Process metrics for a subset of symbols
4. Validate metric outputs
5. Visualize metric distributions
6. Export and save processed metrics

**Data Source:** `/Lean/Data/equity/usa/daily/` (mounted from `C:\Users\kenbr\QC\data` on host)

## Cell 1: Import Required Libraries and Configuration

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import sys
import warnings
warnings.filterwarnings('ignore')

# Add current directory to path to import metrics_processor
sys.path.insert(0, str(Path.cwd()))

# Data configuration
DATA_ROOT = Path("/Lean/Data")
EQUITY_DAILY = DATA_ROOT / "equity" / "usa" / "daily"
PROCESSED_METRICS_DIR = Path("processed_metrics")

# Ensure output directory exists
PROCESSED_METRICS_DIR.mkdir(exist_ok=True)

print(f"[OK] Data root: {DATA_ROOT}")
print(f"[OK] Equity daily data: {EQUITY_DAILY}")
print(f"[OK] Data folder exists: {EQUITY_DAILY.exists()}")

if EQUITY_DAILY.exists():
    zip_count = len(list(EQUITY_DAILY.glob("*.zip")))
    print(f"[OK] Found {zip_count} symbol data files")
    print(f"[OK] Metrics output directory: {PROCESSED_METRICS_DIR.absolute()}")

[OK] Data root: /Lean/Data
[OK] Equity daily data: /Lean/Data/equity/usa/daily
[OK] Data folder exists: True
[OK] Found 560 symbol data files
[OK] Metrics output directory: /Lean/Launcher/bin/Debug/Notebooks/processed_metrics


## Cell 2: Import MetricsProcessor Class

In [11]:
# Import the MetricsProcessor class (with reload during notebook iteration)
import importlib
import metrics_processor as metrics_processor_module
importlib.reload(metrics_processor_module)
MetricsProcessor = metrics_processor_module.MetricsProcessor
print("[OK] Successfully imported MetricsProcessor")

# Show available methods
print("\n[OK] MetricsProcessor methods:")
methods = [m for m in dir(MetricsProcessor) if not m.startswith('_')]
for method in methods:
    print(f"    - {method}")

[OK] Successfully imported MetricsProcessor

[OK] MetricsProcessor methods:
    - collect_metrics
    - get_available_symbols
    - load_symbol_data
    - save_metrics


## Cell 3: Initialize Processor and Get Available Symbols

In [12]:
# Initialize the MetricsProcessor
processor = MetricsProcessor(
    data_root=str(DATA_ROOT),
    output_dir=str(PROCESSED_METRICS_DIR)
)

# Date range configuration (set this here)
END_DATE = datetime(2025, 10, 18)
START_DATE = END_DATE - timedelta(days=3 * 365)

print(f"[OK] MetricsProcessor initialized")
print(f"   Data root: {processor.data_root}")
print(f"   Output dir: {processor.output_dir}")
print(f"   Date range: {START_DATE.date()} to {END_DATE.date()}")

# Get all available symbols
all_symbols = processor.get_available_symbols()
print(f"\n[OK] Total available symbols: {len(all_symbols)}")
print(f"Sample symbols: {all_symbols[:20]}")

[OK] Data root: /Lean/Data
[OK] Output directory: /Lean/Launcher/bin/Debug/Notebooks/processed_metrics
[OK] MetricsProcessor initialized
   Data root: /Lean/Data
   Output dir: processed_metrics
   Date range: 2022-10-19 to 2025-10-18
[OK] Found 560 available symbols

[OK] Total available symbols: 560
Sample symbols: ['A', 'AAA', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AFL', 'AIG']


## Cell 4: Test Data Loading for Individual Symbols

In [13]:
# Test loading data for a few symbols
test_symbols = all_symbols[:5]  # Test with first 5 symbols

print(f"Testing data loading for {len(test_symbols)} symbols...")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}\n")

test_data = {}
for symbol in test_symbols:
    df = processor.load_symbol_data(symbol, START_DATE, END_DATE)
    if df is not None and not df.empty:
        test_data[symbol] = df
        print(f"  {symbol:<8} | Shape: {df.shape} | Date range: {df.index.min().date()} to {df.index.max().date()}")
    else:
        print(f"  {symbol:<8} | [Failed to load]")

# Show sample data
if test_data:
    sample_symbol = list(test_data.keys())[0]
    sample_df = test_data[sample_symbol]
    print(f"\n[OK] Sample data from {sample_symbol}:")
    print(f"Columns: {list(sample_df.columns)}")
    print(f"\nFirst few rows:")
    print(sample_df.head())

Testing data loading for 5 symbols...
Date range: 2022-10-19 to 2025-10-18

  A        | Shape: (752, 6) | Date range: 2022-10-19 to 2025-10-17
  AAA      | [Failed to load]
  AAL      | Shape: (752, 6) | Date range: 2022-10-19 to 2025-10-17
  AAP      | Shape: (752, 6) | Date range: 2022-10-19 to 2025-10-17
  AAPL     | Shape: (752, 6) | Date range: 2022-10-19 to 2025-10-17

[OK] Sample data from A:
Columns: ['time', 'open', 'high', 'low', 'close', 'volume']

First few rows:
                      time        open        high         low       close  \
date                                                                         
2022-10-19  20221019 00:00  127.067445  127.223708  124.264556  125.944344   
2022-10-20  20221020 00:00  124.840736  126.706075  122.672649  122.994934   
2022-10-21  20221021 00:00  123.414882  127.086958  121.901125  126.842804   
2022-10-24  20221024 00:00  128.063569  130.016800  126.774430  129.206207   
2022-10-25  20221025 00:00  129.206186  131.940711 

## Cell 5: Process Metrics for a Test Universe

In [15]:
# Process metrics for a subset of symbols (for testing speed)
import random
random.seed(42)

# Test with different universe sizes
test_universe_size = 20  # Test with 20 symbols first
test_universe = random.sample(all_symbols, min(test_universe_size, len(all_symbols)))

print(f"Processing metrics for {len(test_universe)} symbols...")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"Symbols: {test_universe}\n")

# Process metrics
test_metrics_df = processor.collect_metrics(test_universe, START_DATE, END_DATE)

if test_metrics_df is not None and len(test_metrics_df) > 0:
    processed_symbols = set(test_metrics_df.index.astype(str))
    requested_symbols = set(test_universe)
    skipped_symbols = sorted(requested_symbols - processed_symbols)

    print(f"\n[OK] Metrics processing complete for {len(test_metrics_df)} symbols!")
    print(f"[OK] Skipped symbols: {len(skipped_symbols)}")
    if skipped_symbols:
        print(f"     {', '.join(skipped_symbols)}")
else:
    print("\n[X] Metrics processing returned no results")
    print(f"[X] All requested symbols were skipped: {', '.join(test_universe)}")

Processing metrics for 20 symbols...
Date range: 2022-10-19 to 2025-10-18
Symbols: ['COO', 'ALK', 'JBL', 'HRL', 'GOOAV', 'DECK', 'CMA', 'ZION', 'CCL', 'RL', 'AMGN', 'AMD', 'CFG', 'GIS', 'HAS', 'VLO', 'ALLE', 'FIS', 'RF', 'GLW']


Processing 20 symbols...

[OK] Collected metrics for 19 stocks

[OK] Metrics processing complete for 19 symbols!
[OK] Skipped symbols: 1
     GOOAV


## Cell 6: Load and Validate Processed Metrics

In [ ]:
# Find and load the latest metrics file
metrics_files = sorted(PROCESSED_METRICS_DIR.glob('metrics_*.csv'))

if metrics_files:
    latest_metrics_file = metrics_files[-1]
    print(f"[OK] Found {len(metrics_files)} metrics files")
    print(f"[OK] Loading latest: {latest_metrics_file.name}")
    
    metrics_df = pd.read_csv(latest_metrics_file, index_col=0)
    print(f"\n[OK] Loaded metrics for {len(metrics_df)} symbols")
    
    # Display basic info
    print(f"\nColumns: {list(metrics_df.columns)}")
    print(f"\nData types:")
    print(metrics_df.dtypes)
    
    # Show sample data
    print(f"\nSample metrics (first 10 symbols):")
    print(metrics_df.head(10))
else:
    print("[X] No metrics files found")
    metrics_df = None

## Cell 7: Validate Metric Outputs

In [ ]:
if metrics_df is not None:
    print("[OK] METRIC VALIDATION REPORT")
    print("=" * 80)
    
    # Check for NaN values
    print("\nNaN Check:")
    nan_counts = metrics_df.isna().sum()
    if nan_counts.sum() == 0:
        print("  [OK] No NaN values found")
    else:
        print("  [!] NaN values detected:")
        for col, count in nan_counts[nan_counts > 0].items():
            print(f"      {col}: {count} NaNs")
    
    # Metric ranges
    print("\nMetric Ranges:")
    for col in metrics_df.columns:
        min_val = metrics_df[col].min()
        max_val = metrics_df[col].max()
        mean_val = metrics_df[col].mean()
        print(f"  {col:<15} | Min: {min_val:>10.4f} | Max: {max_val:>10.4f} | Mean: {mean_val:>10.4f}")
    
    # Verify expected metric properties
    print("\n[OK] Metric Property Checks:")
    if 'momentum' in metrics_df.columns:
        print(f"  Momentum range: [{metrics_df['momentum'].min():.4f}, {metrics_df['momentum'].max():.4f}]")
    if 'volatility' in metrics_df.columns:
        print(f"  Volatility range: [{metrics_df['volatility'].min():.4f}, {metrics_df['volatility'].max():.4f}]")
        if metrics_df['volatility'].min() >= 0:
            print(f"      [OK] Volatility is always non-negative")
    if 'price' in metrics_df.columns:
        if metrics_df['price'].min() > 0:
            print(f"      [OK] All prices are positive")
    
    # Statistical summary
    print("\n[OK] Statistical Summary:")
    print(metrics_df.describe())
else:
    print("[X] No metrics to validate")

## Cell 8: Visualize Metric Distributions

In [ ]:
if metrics_df is not None and len(metrics_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f'Metric Distributions ({len(metrics_df)} symbols)', fontsize=14, fontweight='bold')
    
    col_idx = 0
    for i in range(2):
        for j in range(3):
            if col_idx < len(metrics_df.columns):
                col_name = metrics_df.columns[col_idx]
                ax = axes[i, j]
                
                # Plot histogram
                ax.hist(metrics_df[col_name].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
                ax.axvline(metrics_df[col_name].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {metrics_df[col_name].mean():.4f}')
                ax.axvline(metrics_df[col_name].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {metrics_df[col_name].median():.4f}')
                
                ax.set_xlabel(col_name, fontsize=11)
                ax.set_ylabel('Frequency', fontsize=11)
                ax.set_title(f'{col_name} Distribution', fontsize=12, fontweight='bold')
                ax.legend(fontsize=9)
                ax.grid(True, alpha=0.3)
                
                col_idx += 1
            else:
                axes[i, j].axis('off')
    
    plt.tight_layout()
    plt.show()
    print("[OK] Distribution plots generated")
else:
    print("[X] No metrics to visualize")

## Cell 9: Correlation Analysis

In [ ]:
if metrics_df is not None and len(metrics_df) > 2:
    # Select numeric columns only
    numeric_cols = metrics_df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 1:
        # Calculate correlation matrix
        correlation_matrix = metrics_df[numeric_cols].corr()
        
        # Plot correlation heatmap
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
                    square=True, linewidths=1, cbar_kws={"shrink": 0.8},
                    fmt='.3f', vmin=-1, vmax=1)
        plt.title('Metric Correlation Matrix', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        # Print correlation matrix
        print("[OK] Metric Correlations:")
        print(correlation_matrix)
    else:
        print("[!] Insufficient numeric columns for correlation analysis")
else:
    print("[X] No metrics to analyze")

## Cell 10: Process Full Universe and Save

In [16]:
# Process metrics for the full universe (all available symbols)
print(f"Processing metrics for full universe ({len(all_symbols)} symbols)...")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")
print("This may take a few minutes...\n")

import time
start_time = time.time()

# Collect metrics for all symbols
full_metrics_df = processor.collect_metrics(all_symbols, START_DATE, END_DATE)

elapsed_time = time.time() - start_time
print(f"\n[OK] Processing complete!")
print(f"[OK] Time elapsed: {elapsed_time:.2f} seconds")
print(f"[OK] Average time per symbol: {elapsed_time / len(all_symbols):.4f} seconds")

if full_metrics_df is not None and len(full_metrics_df) > 0:
    processed_symbols = set(full_metrics_df.index.astype(str))
    requested_symbols = set(all_symbols)
    skipped_symbols = sorted(requested_symbols - processed_symbols)

    saved_file = processor.save_metrics(full_metrics_df, END_DATE)
    print(f"\n[OK] Saved metrics for {len(full_metrics_df)} symbols to:")
    print(f"     {saved_file.absolute()}")
    print(f"[OK] Skipped symbols: {len(skipped_symbols)}")
    if skipped_symbols:
        preview = skipped_symbols[:25]
        print(f"     First {len(preview)} skipped: {', '.join(preview)}")
else:
    print("\n[X] No full-universe metrics to save")
    print("[X] All symbols were skipped")

Processing metrics for full universe (560 symbols)...
Date range: 2022-10-19 to 2025-10-18
This may take a few minutes...


Processing 560 symbols...
  Progress: 50/560
  Progress: 100/560
  Progress: 150/560
  Progress: 200/560
  Progress: 250/560
  Progress: 300/560
  Progress: 350/560
  Progress: 400/560
  Progress: 450/560
  Progress: 500/560
  Progress: 550/560

[OK] Collected metrics for 547 stocks

[OK] Processing complete!
[OK] Time elapsed: 20.28 seconds
[OK] Average time per symbol: 0.0362 seconds
[OK] Metrics saved to: /Lean/Launcher/bin/Debug/Notebooks/processed_metrics/metrics_20251018_20260308_235716.csv
[OK] Metadata saved to: /Lean/Launcher/bin/Debug/Notebooks/processed_metrics/metadata.json
[OK] Index updated: /Lean/Launcher/bin/Debug/Notebooks/processed_metrics/index.csv

[OK] Saved metrics for 547 symbols to:
     /Lean/Launcher/bin/Debug/Notebooks/processed_metrics/metrics_20251018_20260308_235716.csv
[OK] Skipped symbols: 13
     First 13 skipped: AAA, BNO, EEM, FB

## Cell 11: Export and Save Processed Metrics

In [17]:
# Check for processed metrics files
metrics_files = sorted(PROCESSED_METRICS_DIR.glob('metrics_*.csv'))
index_files = sorted(PROCESSED_METRICS_DIR.glob('index_*.csv'))
metadata_files = sorted(PROCESSED_METRICS_DIR.glob('metadata_*.json'))

print("[OK] PROCESSED METRICS FILES")
print("=" * 80)

if metrics_files:
    print(f"\n[OK] Metrics files ({len(metrics_files)} total):")
    for f in metrics_files[-3:]:
        file_size_mb = f.stat().st_size / (1024 * 1024)
        print(f"     {f.name} ({file_size_mb:.2f} MB)")
else:
    print(f"\n[!] No metrics files found")

if index_files:
    print(f"\n[OK] Index files ({len(index_files)} total):")
    for f in index_files[-3:]:
        print(f"     {f.name}")

if metadata_files:
    print(f"\n[OK] Metadata files ({len(metadata_files)} total):")
    for f in metadata_files[-3:]:
        print(f"     {f.name}")
        # Show metadata content
        import json
        with open(f, 'r') as mf:
            metadata = json.load(mf)
            print(f"        Symbols processed: {metadata.get('symbols_processed', 'N/A')}")
            print(f"        Timestamp: {metadata.get('timestamp', 'N/A')}")

print(f"\n[OK] Output directory: {PROCESSED_METRICS_DIR.absolute()}")

[OK] PROCESSED METRICS FILES

[OK] Metrics files (1 total):
     metrics_20251018_20260308_235716.csv (0.06 MB)

[OK] Output directory: /Lean/Launcher/bin/Debug/Notebooks/processed_metrics


## Cell 12: Summary and Next Steps

In [18]:
print("[OK] METRICS PROCESSOR RESEARCH SUMMARY")
print("=" * 80)

print("\n✓ COMPLETED TASKS:")
print("  1. Loaded MetricsProcessor class")
print("  2. Tested data loading for individual symbols")
print("  3. Processed metrics for test universe")
print("  4. Validated metric outputs (no NaN values, correct ranges)")
print("  5. Generated distribution visualizations")
print("  6. Calculated correlation matrix")
print("  7. Processed full universe metrics")
print("  8. Saved metrics to CSV with metadata")

print("\n→ NEXT STEPS:")
print("  • Use research_local.ipynb to load these pre-processed metrics")
#  • Implement clustering logic in clustering.py")
print("  • Create clustering_research.ipynb for cluster testing")
print("  • Create visualization.py for regime visualization")
print("  • Create main_pipeline.ipynb to orchestrate full workflow")

print("\n[OK] Metrics processor research complete!")

[OK] METRICS PROCESSOR RESEARCH SUMMARY

✓ COMPLETED TASKS:
  1. Loaded MetricsProcessor class
  2. Tested data loading for individual symbols
  3. Processed metrics for test universe
  4. Validated metric outputs (no NaN values, correct ranges)
  5. Generated distribution visualizations
  6. Calculated correlation matrix
  7. Processed full universe metrics
  8. Saved metrics to CSV with metadata

→ NEXT STEPS:
  • Use research_local.ipynb to load these pre-processed metrics
  • Create clustering_research.ipynb for cluster testing
  • Create visualization.py for regime visualization
  • Create main_pipeline.ipynb to orchestrate full workflow

[OK] Metrics processor research complete!
